In [1]:
# ═══════════════════════════════════════════════════════
# CELL 1 – Imports, paths, configuration (FULL FINE-TUNE, no LoRA)
# ═══════════════════════════════════════════════════════
import json, os, random, time, unicodedata, warnings
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import editdistance
from transformers import VisionEncoderDecoderModel, TrOCRProcessor, get_scheduler
from tqdm import tqdm

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Paths (reuse the same dataset/vocab as the LoRA run) ──────────────
DATASET_DIR    = Path("/kaggle/input/datasets/nehamalik10/hindi-ocr-new-dataset")
ARTIFACTS_PATH = Path("/kaggle/input/datasets/nehamalik10/phase4-session2-outputs/session2_artifacts.json")
OUTPUT_DIR     = Path("/kaggle/working/trocr_base_fullft")
CKPT_DIR       = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ─────────────────────────────────────
MODEL_NAME     = "microsoft/trocr-base-handwritten"
MAX_TARGET_LEN = 32
TROCR_SIZE     = 384

# Full fine-tune of a ~334M-param model needs more memory headroom than
# LoRA did (optimizer states for ALL params, not just adapters). Smaller
# micro-batch, more grad accumulation, same effective batch as before.
BATCH_SIZE   = 4
GRAD_ACCUM   = 8          # effective batch size = 32
NUM_WORKERS  = 2

# Two-phase schedule — same philosophy as the proven small-model run:
#   Phase A: only the (tied) embedding/output-head trains. Everything
#            else frozen. Gives ~130 new Devanagari token embeddings a
#            stable target to learn before the whole network starts
#            moving — prevents large random-init embedding gradients
#            from destabilizing the pretrained encoder/decoder weights.
#   Phase B: EVERYTHING unfrozen — encoder AND decoder, full parameters,
#            no LoRA, no rank constraint. This is the actual fix: let
#            the visual backbone genuinely relearn Devanagari-appropriate
#            features instead of offsetting a frozen English-handwriting
#            backbone within a low-rank subspace.
EPOCHS_PHASE_A = 2
EPOCHS_PHASE_B = 16        # generous budget — full FT typically needs
                           # more epochs to fully converge than LoRA

LR_PHASE_A       = 4e-5    # embed/head only
LR_PHASE_B_MAIN  = 2e-5    # encoder + decoder body
LR_PHASE_B_EMBED = 1e-5    # embed/head, slightly lower once the whole
                           # network is moving together
WEIGHT_DECAY     = 0.01
WARMUP_STEPS     = 500
GRAD_CLIP        = 1.0

EVAL_EVERY_STEPS   = 1500
EVAL_SAMPLES_QUICK = 600
EVAL_SAMPLES_FULL  = 2000
NUM_BEAMS_EVAL     = 4
NUM_BEAMS_TEST     = 8
EARLY_STOP_PATIENCE = 5    # slightly more lenient than the LoRA run —
                           # full FT can plateau then improve, don't
                           # want to stop prematurely on a bigger model

print("Configuration ready.")
print(f"  Phase A: {EPOCHS_PHASE_A} epochs (embed/head only, LR={LR_PHASE_A})")
print(f"  Phase B: {EPOCHS_PHASE_B} epochs (FULL unfreeze, LR_main={LR_PHASE_B_MAIN}, LR_embed={LR_PHASE_B_EMBED})")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
Configuration ready.
  Phase A: 2 epochs (embed/head only, LR=4e-05)
  Phase B: 16 epochs (FULL unfreeze, LR_main=2e-05, LR_embed=1e-05)
  Effective batch size: 32


In [2]:
# ═══════════════════════════════════════════════════════
# CELL 2 – Load vocabulary (exactly as the small model / LoRA run)
# ═══════════════════════════════════════════════════════
with open(ARTIFACTS_PATH, "r", encoding="utf-8") as f:
    art = json.load(f)

char_to_token = art["char_to_token"]
token_to_char = {int(k): v for k, v in art["token_to_char"].items()}
VOCAB_SIZE = art["VOCAB_SIZE"]
PAD_ID = art["PAD_ID"]
BOS_ID = art["BOS_ID"]
EOS_ID = art["EOS_ID"]
UNK_ID = art["UNK_ID"]

print(f"Vocabulary loaded: {VOCAB_SIZE} tokens")
print(f"Special tokens: PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}, UNK={UNK_ID}")

def encode(text):
    text = unicodedata.normalize("NFC", text)
    ids = [char_to_token.get(ch, UNK_ID) for ch in text]
    ids.append(EOS_ID)
    return ids

def decode_ids(ids):
    chars = []
    for tid in ids:
        if isinstance(tid, torch.Tensor):
            tid = tid.item()
        if tid == EOS_ID or tid == PAD_ID:
            break
        if tid == BOS_ID:
            continue
        if tid in token_to_char:
            chars.append(token_to_char[tid])
    return "".join(chars)

test_text = "हिन्दी भारत"
enc = encode(test_text)
dec = decode_ids(enc)
print(f"Test roundtrip: '{test_text}' -> '{dec}' ({'OK' if test_text == dec else 'FAIL'})")

Vocabulary loaded: 140 tokens
Special tokens: PAD=0, BOS=1, EOS=2, UNK=3
Test roundtrip: 'हिन्दी भारत' -> 'हिन्दी भारत' (OK)


In [3]:
# ═══════════════════════════════════════════════════════
# CELL 3 – Dataset & preprocessing (proven Phase 3.5 pipeline, unchanged)
# ═══════════════════════════════════════════════════════
train_df = pd.read_parquet(DATASET_DIR / "train.parquet")
val_df   = pd.read_parquet(DATASET_DIR / "val.parquet")
test_df  = pd.read_parquet(DATASET_DIR / "test.parquet")
print(f"Data: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

IMG_COL, LABEL_COL = "image", "text"
TIGHT_CROP_PAD = 2

def row_to_image(row):
    value = row[IMG_COL]
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return Image.open(BytesIO(value["bytes"])).convert("RGB")
        if value.get("path") is not None:
            return Image.open(value["path"]).convert("RGB")
    if isinstance(value, (bytes, bytearray)):
        return Image.open(BytesIO(value)).convert("RGB")
    if isinstance(value, np.ndarray):
        return Image.fromarray(value).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(value)}")

def preprocess_for_trocr(pil_img):
    gray = np.array(pil_img.convert("L"))
    p2, p98 = np.percentile(gray, (2, 98))
    if p98 > p2:
        gray = np.clip((gray - p2) / (p98 - p2) * 255, 0, 255).astype(np.uint8)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = cv2.findNonZero(bw)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        pad = TIGHT_CROP_PAD
        y1, y2 = max(0, y - pad), min(gray.shape[0], y + h + pad)
        x1, x2 = max(0, x - pad), min(gray.shape[1], x + w + pad)
        gray = gray[y1:y2, x1:x2]
    if gray.shape[0] < 4 or gray.shape[1] < 4:
        gray = np.full((32, 128), 255, dtype=np.uint8)
    rgb = Image.fromarray(cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB))

    w, h = rgb.size
    scale = min(TROCR_SIZE / w, TROCR_SIZE / h)
    nw, nh = max(1, int(w * scale)), max(1, int(h * scale))
    resized = rgb.resize((nw, nh), Image.LANCZOS)
    canvas = Image.new("RGB", (TROCR_SIZE, TROCR_SIZE), (255, 255, 255))
    canvas.paste(resized, ((TROCR_SIZE - nw) // 2, (TROCR_SIZE - nh) // 2))
    return canvas

def aug_rotate(img):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), random.uniform(-5, 5), 1.0)
    return cv2.warpAffine(img, M, (w, h), borderValue=(255,255,255))
def aug_noise(img):
    n = np.random.normal(0, random.uniform(5, 15), img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + n, 0, 255).astype(np.uint8)
def aug_erode_dilate(img):
    k = np.ones((2, 2), np.uint8)
    return cv2.erode(img, k, iterations=1) if random.random() < 0.5 else cv2.dilate(img, k, iterations=1)
def aug_shear(img):
    h, w = img.shape[:2]
    M = np.float32([[1, random.uniform(-0.2, 0.2), 0], [0, 1, 0]])
    return cv2.warpAffine(img, M, (w, h), borderValue=(255,255,255))
def aug_brightness(img):
    return np.clip(img.astype(np.float32)*random.uniform(0.7,1.3)+random.uniform(-20,20), 0, 255).astype(np.uint8)
def aug_elastic(img, alpha=30, sigma=5):
    h, w = img.shape[:2]
    dx = cv2.GaussianBlur((np.random.rand(h,w)*2-1).astype(np.float32), (0,0), sigma) * alpha
    dy = cv2.GaussianBlur((np.random.rand(h,w)*2-1).astype(np.float32), (0,0), sigma) * alpha
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    map_x = np.float32(x + dx); map_y = np.float32(y + dy)
    if len(img.shape) == 3:
        return cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR, borderValue=(255,255,255))
    return cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR, borderValue=255)
def aug_blur(img):
    k = random.choice([3, 5])
    return cv2.GaussianBlur(img, (k, k), 0)
def aug_cutout(img):
    h, w = img.shape[:2]
    ch, cw = random.randint(h//12,h//6), random.randint(w//12,w//6)
    y, x = random.randint(0,h-ch), random.randint(0,w-cw)
    c = img.copy(); c[y:y+ch, x:x+cw] = 255; return c

def augment_image(pil_img):
    img = np.array(pil_img)
    if random.random() < 0.5:  img = aug_rotate(img)
    if random.random() < 0.4:  img = aug_noise(img)
    if random.random() < 0.2:  img = aug_erode_dilate(img)
    if random.random() < 0.4:  img = aug_shear(img)
    if random.random() < 0.35: img = aug_brightness(img)
    if random.random() < 0.15: img = aug_cutout(img)
    if random.random() < 0.3:  img = aug_elastic(img)
    if random.random() < 0.2:  img = aug_blur(img)
    return Image.fromarray(img)

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)

class WordDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = unicodedata.normalize("NFC", str(row[LABEL_COL]))
        try:
            pil_img = row_to_image(row)
            pil_img = preprocess_for_trocr(pil_img)
        except Exception:
            pil_img = Image.new("RGB", (TROCR_SIZE, TROCR_SIZE), color=255)
            text = ""
        if self.augment and text:
            pil_img = augment_image(pil_img)
        pixel_values = processor(images=pil_img, return_tensors="pt").pixel_values.squeeze(0)
        label_ids = encode(text)
        if len(label_ids) > MAX_TARGET_LEN:
            label_ids = label_ids[:MAX_TARGET_LEN - 1] + [EOS_ID]
        padded = label_ids + [PAD_ID] * (MAX_TARGET_LEN - len(label_ids))
        labels = torch.tensor(padded, dtype=torch.long)
        labels[labels == PAD_ID] = -100
        return {"pixel_values": pixel_values, "labels": labels}

def collate_fn(batch):
    pv = torch.stack([x["pixel_values"] for x in batch])
    labels = torch.stack([x["labels"] for x in batch])
    return {"pixel_values": pv, "labels": labels}

train_ds = WordDataset(train_df, augment=True)
val_ds   = WordDataset(val_df, augment=False)
test_ds  = WordDataset(test_df, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
print(f"Loaders ready: train={len(train_loader)} batches, val={len(val_loader)}, test={len(test_loader)}")

Data: train=150,000  val=20,000  test=30,000


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Loaders ready: train=37500 batches, val=5000, test=7500


In [4]:
# ═══════════════════════════════════════════════════════
# CELL 4 – Model setup: FULL FINE-TUNE (no LoRA, no gradient checkpointing)
# ═══════════════════════════════════════════════════════
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

model.config.pad_token_id = PAD_ID
model.config.decoder_start_token_id = BOS_ID
model.config.eos_token_id = EOS_ID
model.config.bos_token_id = BOS_ID
model.decoder.config.pad_token_id = PAD_ID
model.decoder.config.bos_token_id = BOS_ID
model.decoder.config.eos_token_id = EOS_ID
model.decoder.config.decoder_start_token_id = BOS_ID

# ── Resize token embeddings for the custom Devanagari vocab ───────────
old_embed = model.decoder.get_input_embeddings().weight.data.clone()
embed_mean = old_embed.mean(dim=0)
model.decoder.resize_token_embeddings(VOCAB_SIZE)
with torch.no_grad():
    new_embed = model.decoder.get_input_embeddings().weight
    for i in range(VOCAB_SIZE):
        if i < old_embed.shape[0]:
            new_embed[i] = old_embed[i]
        else:
            new_embed[i] = embed_mean + torch.randn_like(embed_mean) * 0.02

embed_ptr = model.decoder.get_input_embeddings().weight.data_ptr()
head_ptr  = model.decoder.get_output_embeddings().weight.data_ptr()
IS_TIED = (embed_ptr == head_ptr)
print(f"Embedding/output-head tied: {IS_TIED}")
if not IS_TIED:
    with torch.no_grad():
        model.decoder.get_output_embeddings().weight.copy_(new_embed)

# ── Phase A starts: EVERYTHING frozen except embed/head ───────────────
for p in model.parameters():
    p.requires_grad = False
model.decoder.get_input_embeddings().weight.requires_grad = True
if not IS_TIED:
    model.decoder.get_output_embeddings().weight.requires_grad = True

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable (Phase A start): {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
print(f"Total model parameters: {total:,}  (~{total/1e6:.0f}M)")

def build_optimizer_phase_a(model):
    embed_params = [model.decoder.get_input_embeddings().weight]
    return optim.AdamW([{"params": embed_params, "lr": LR_PHASE_A, "weight_decay": 0.0}])

def build_optimizer_phase_b(model):
    for p in model.parameters():
        p.requires_grad = True
    embed_weight = model.decoder.get_input_embeddings().weight
    special_ids = {id(embed_weight)}
    embed_params = [embed_weight]
    if not IS_TIED:
        out_weight = model.decoder.get_output_embeddings().weight
        special_ids.add(id(out_weight))
        embed_params.append(out_weight)
    main_params = [p for p in model.parameters() if id(p) not in special_ids]
    groups = [
        {"params": main_params, "lr": LR_PHASE_B_MAIN, "weight_decay": WEIGHT_DECAY},
        {"params": embed_params, "lr": LR_PHASE_B_EMBED, "weight_decay": 0.0},
    ]
    return optim.AdamW(groups)

scaler = GradScaler("cuda")
print("Model ready. Phase A: embed/head only, everything else frozen.")

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding/output-head tied: True
Trainable (Phase A start): 143,360 / 282,593,792 (0.051%)
Total model parameters: 282,593,792  (~283M)
Model ready. Phase A: embed/head only, everything else frozen.


In [5]:
# ═══════════════════════════════════════════════════════
# SELF‑CONTAINED RESUME – Phase B for 4 more epochs
# ═══════════════════════════════════════════════════════
import torch, time
from tqdm import tqdm

RESUME_EPOCHS = 4
LOAD_CKPT_PATH = Path("/kaggle/input/models/nehamalik10/trocr-base-1/pytorch/default/1/trocr 1.pt")   # ← adjust if needed

if not LOAD_CKPT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found at {LOAD_CKPT_PATH}")

ckpt = torch.load(LOAD_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(device)

model.config.use_cache = False
model.decoder.config.use_cache = False

# Full unfreeze (Phase B)
for p in model.parameters():
    p.requires_grad = True
optimizer_b = build_optimizer_phase_b(model)

# ── Helper functions (same as in Cell 5) ────────────────
@torch.no_grad()
def evaluate(model, loader, max_samples=EVAL_SAMPLES_FULL, num_beams=1):
    model.eval()
    total_cer, total_wer, count = 0.0, 0.0, 0
    for batch in loader:
        if count >= max_samples: break
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]
        generated = model.generate(
            pixel_values, max_length=MAX_TARGET_LEN,
            num_beams=num_beams, decoder_start_token_id=BOS_ID,
        )
        for i in range(pixel_values.size(0)):
            if count >= max_samples: break
            pred_text = decode_ids(generated[i])
            gt_ids = labels[i][labels[i] != -100].tolist()
            gt_text = decode_ids(gt_ids)
            if len(gt_text) == 0:
                count += 1
                continue
            ed = editdistance.eval(pred_text, gt_text)
            total_cer += ed / max(len(gt_text), 1)
            pred_words = pred_text.split(); gt_words = gt_text.split()
            if len(gt_words) > 0:
                total_wer += editdistance.eval(pred_words, gt_words) / len(gt_words)
            else:
                total_wer += (0.0 if len(pred_words) == 0 else 1.0)
            count += 1
    model.train()
    return total_cer / max(count, 1), total_wer / max(count, 1), count

def save_checkpoint(model, epoch, phase, val_cer, val_wer, tag="best_model.pt"):
    torch.save({
        "epoch": epoch, "phase": phase,
        "model_state_dict": model.state_dict(),
        "val_cer": val_cer, "val_wer": val_wer,
        "vocab_size": VOCAB_SIZE,
        "char_to_token": char_to_token,
        "token_to_char": {str(k): v for k, v in token_to_char.items()},
    }, CKPT_DIR / tag)

def run_phase(model, optimizer, phase_name, n_epochs, start_epoch):
    global best_cer, epochs_without_improvement
    total_steps = (n_epochs * len(train_loader)) // GRAD_ACCUM
    scheduler = get_scheduler("cosine", optimizer=optimizer,
                               num_warmup_steps=0,
                               num_training_steps=max(1, total_steps))

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*60}\nPHASE {phase_name}: {n_epochs} epochs | trainable: {trainable:,}\n{'='*60}")

    global_step = 0
    for epoch in range(1, n_epochs + 1):
        global_ep = start_epoch + epoch - 1
        model.train()
        epoch_loss, n_batches = 0.0, 0
        t0 = time.time()

        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f"[{phase_name}] Epoch {epoch}/{n_epochs}", unit="batch")
        for batch_idx, batch in pbar:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            with autocast("cuda"):
                outputs = model(pixel_values=pixel_values, labels=labels)
                loss = outputs.loss / GRAD_ACCUM

            scaler.scale(loss).backward()

            if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1

                if global_step % EVAL_EVERY_STEPS == 0:
                    q_cer, q_wer, _ = evaluate(model, val_loader, max_samples=EVAL_SAMPLES_QUICK, num_beams=1)
                    pbar.write(f"    [step {global_step}] quick val CER={q_cer:.4f} WER={q_wer:.4f}")
                    if q_cer < best_cer:
                        best_cer = q_cer
                        save_checkpoint(model, global_ep, phase_name, q_cer, q_wer)
                        pbar.write(f"    >> Best model saved (CER={q_cer:.4f}) [mid-epoch]")

            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            pbar.set_postfix(loss=f"{loss.item()*GRAD_ACCUM:.4f}")

        avg_loss = epoch_loss / max(n_batches, 1)
        val_cer, val_wer, _ = evaluate(model, val_loader, max_samples=EVAL_SAMPLES_FULL, num_beams=NUM_BEAMS_EVAL)
        lr_report = optimizer.param_groups[-1]["lr"]
        elapsed = time.time() - t0
        print(f"  Epoch {global_ep} [{phase_name}] | loss={avg_loss:.4f} | "
              f"val CER={val_cer:.4f} | val WER={val_wer:.4f} | LR={lr_report:.2e} | time={elapsed:.0f}s")

        if val_cer < best_cer:
            best_cer = val_cer
            epochs_without_improvement = 0
            save_checkpoint(model, global_ep, phase_name, val_cer, val_wer)
            print(f"    >> Best model saved (CER={val_cer:.4f})")
        else:
            epochs_without_improvement += 1
            print(f"    No improvement ({epochs_without_improvement}/{EARLY_STOP_PATIENCE})")
            if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                print("    Early stopping triggered.")
                return global_ep + 1, True

    return start_epoch + n_epochs, False

# ── Resume training ─────────────────────────────────────
best_cer = ckpt["val_cer"]
epochs_without_improvement = 0
resume_epoch = ckpt["epoch"] + 1
print(f"Resumed from epoch {ckpt['epoch']}, phase {ckpt['phase']}, val CER={best_cer:.4f}")

next_epoch, stopped = run_phase(model, optimizer_b, "B", RESUME_EPOCHS, start_epoch=resume_epoch)
print(f"\nSession finished at epoch {next_epoch - 1}. Best val CER so far: {best_cer:.4f}")

Resumed from epoch 5, phase B, val CER=0.0164

PHASE B: 4 epochs | trainable: 282,593,792


[B] Epoch 1/4:  32%|███▏      | 12001/37500 [40:32<96:30:51, 13.63s/batch, loss=0.0339] 

    [step 1500] quick val CER=0.0372 WER=0.1133


[B] Epoch 1/4:  64%|██████▍   | 24001/37500 [1:21:02<50:20:18, 13.42s/batch, loss=0.0346]

    [step 3000] quick val CER=0.0297 WER=0.0950


...

    [step 16500] quick val CER=0.0127 WER=0.0483


[B] Epoch 4/4:  84%|████████▍ | 31489/37500 [1:46:31<22:28:01, 13.46s/batch, loss=0.1455]

    [step 18000] quick val CER=0.0132 WER=0.0483


[B] Epoch 4/4: 100%|██████████| 37500/37500 [2:06:17<00:00,  4.95batch/s, loss=0.1553]   


  Epoch 9 [B] | loss=0.0666 | val CER=0.0165 | val WER=0.0545 | LR=2.81e-13 | time=8199s
    No improvement (4/5)

Session finished at epoch 9. Best val CER so far: 0.0121


In [ ]:
# ═══════════════════════════════════════════════════════
# RESUME Phase B (no gradient checkpointing)
# ═══════════════════════════════════════════════════════
RESUME_EPOCHS = 1   # how many more epochs to run this session

LOAD_CKPT_PATH = CKPT_DIR / "best_model.pt"
if not LOAD_CKPT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found at {LOAD_CKPT_PATH}")

ckpt = torch.load(LOAD_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(device)

# ── Important: keep cache disabled for training, but NO checkpointing ──
model.config.use_cache = False
model.decoder.config.use_cache = False

# Full unfreeze (Phase B)
for p in model.parameters():
    p.requires_grad = True
optimizer_b = build_optimizer_phase_b(model)

best_cer = ckpt["val_cer"]
resume_epoch = ckpt["epoch"] + 1
epochs_without_improvement = 0
print(f"Resumed from epoch {ckpt['epoch']}, phase {ckpt['phase']}, val CER={best_cer:.4f}")

next_epoch, stopped = run_phase(model, optimizer_b, "B", RESUME_EPOCHS, start_epoch=resume_epoch)
print(f"\nSession finished at epoch {next_epoch - 1}. Best val CER so far: {best_cer:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 7 – Final evaluation on test set (beam search)
# ═══════════════════════════════════════════════════════
ckpt = torch.load(CKPT_DIR / "best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded best model (epoch {ckpt['epoch']}, phase {ckpt['phase']}, val CER={ckpt['val_cer']:.4f})")

test_cer, test_wer, n_test = evaluate(model, test_loader, max_samples=len(test_ds), num_beams=NUM_BEAMS_TEST)
print(f"\nTest set ({n_test} samples), beam={NUM_BEAMS_TEST}:")
print(f"  CER: {test_cer:.4f}")
print(f"  WER: {test_wer:.4f}")
print(f"  Char accuracy: {(1-test_cer)*100:.2f}%")
print(f"  Word accuracy: {(1-test_wer)*100:.2f}%")